# Random Forest — Ceftazidime Resistance in *Escherichia coli*

Species-specific, drug-specific Random Forest classifier for AMR prediction
from MALDI-TOF binned spectra.

## Scope

- **Drug:** Ceftazidime
- **Pathogen:** *Escherichia coli*
- **Data:** DRIAMS A, B, C, D (processed, 6000 bins at 3 Da)

## What this notebook covers

1. Load Ceftazidime data from all 4 sites, filter to E. coli
2. **Tune once** — GridSearchCV on pooled A+B+C+D to find best RF hyperparameters
3. **Cross-site evaluation** — Train on DRIAMS-A, test on B, C, D separately
4. **Multi-seed pooled evaluation** — 5× random 75/25 splits on pooled A+B+C+D
5. **Feature importance** — Which m/z bins drive the resistance prediction?
6. Report generation

In [ ]:
# All core dependencies (numpy, pandas, sklearn, matplotlib)
# come pre-installed on Google Colab.
!pip install seaborn --quiet


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False


In [ ]:
# =============================================================================
# 1. IMPORTS
# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# ── Paths ──
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

SITES = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / "Ceftazidime" / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / "Ceftazidime" / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / "Ceftazidime" / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / "Ceftazidime" / "data.csv",
}

OUT_DIR = Path("./results_rf")
OUT_DIR.mkdir(exist_ok=True)

DRUG = "Ceftazidime"
SPECIES = "Escherichia coli"

print(f"DRYAD:       {DRYAD.resolve()}")
print(f"Colab mode:  {IN_COLAB}")
print(f"Output to:   {OUT_DIR.resolve()}")
print(f"Drug:        {DRUG}")
print(f"Species:     {SPECIES}")
for s, p in SITES.items():
    status = "EXISTS" if p.exists() else "MISSING"
    print(f"  DRIAMS-{s}: {status} — {p}")


---
## 2. LOAD & FILTER DATA

In [ ]:
# =============================================================================
# 2.1  Load each site, filter to E. coli, collect metadata
# =============================================================================

site_data = {}
mz_axis = np.arange(2000, 2000 + 6000 * 3, 3)  # 3 Da bins from 2000 Da

for site, path in SITES.items():
    df = pd.read_csv(path)
    df_eco = df[df["species"] == SPECIES].copy()

    # Separate features (6000 bins) and labels
    bin_cols = [c for c in df_eco.columns if c.startswith("bin_")]
    X = df_eco[bin_cols].to_numpy(dtype="float32")
    y = df_eco["label"].to_numpy(dtype="int64")

    site_data[site] = {"X": X, "y": y, "n": len(y)}
    r = (y == 1).sum()
    s = (y == 0).sum()

    print(f"DRIAMS-{site}: n={len(y):5d}  R={r:5d}  S={s:5d}  %R={r/len(y)*100:5.1f}%")

# Global totals
total_n = sum(d["n"] for d in site_data.values())
total_r = sum((d["y"] == 1).sum() for d in site_data.values())
print(f"\nTOTAL:    n={total_n:5d}  R={total_r:5d}  S={total_n-total_r:5d}  %R={total_r/total_n*100:5.1f}%")

---
## 3. TUNE ONCE — GridSearchCV on Pooled A+B+C+D

Hyperparameter sweep on the pooled data to find the best Random Forest configuration. These params will be used for all subsequent evaluations.

In [ ]:
# =============================================================================
# 3.1  Pool all sites
# =============================================================================

# Concatenate all sites
X_pooled = np.vstack([site_data[s]["X"] for s in "ABCD"])
y_pooled = np.hstack([site_data[s]["y"] for s in "ABCD"])

# Also store site labels for cross-site analysis
site_pooled = np.hstack([
    np.full(site_data[s]["n"], s) for s in "ABCD"
])

print(f"Pooled data: {X_pooled.shape[0]} samples × {X_pooled.shape[1]} bins")
print(f"  Class distribution: R={(y_pooled==1).sum()}  S={(y_pooled==0).sum()}")

# =============================================================================
# 3.2  Single stratified split for GridSearch
# =============================================================================

SEED_TUNE = 42
X_tr, X_te, y_tr, y_te = train_test_split(
    X_pooled, y_pooled, test_size=0.25,
    stratify=y_pooled, random_state=SEED_TUNE
)
print(f"\nTuning split: train={len(y_tr)}  test={len(y_te)}")

# =============================================================================
# 3.3  GridSearchCV
# =============================================================================

param_grid = {
    "n_estimators":      [100, 300, 500],
    "max_depth":         [10, 20, 30, None],
    "min_samples_leaf":  [2, 5, 10],
    "class_weight":      ["balanced", "balanced_subsample"],
}

print("\nGridSearchCV — Random Forest:")
print(f"  n_estimators:      {param_grid['n_estimators']}")
print(f"  max_depth:         {param_grid['max_depth']}")
print(f"  min_samples_leaf:  {param_grid['min_samples_leaf']}")
print(f"  class_weight:      {param_grid['class_weight']}")
tot = np.prod([len(v) for v in param_grid.values()])
print(f"  Total combinations: {tot}")

grid = GridSearchCV(
    RandomForestClassifier(oob_score=True, random_state=SEED_TUNE, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring="balanced_accuracy",
    verbose=1,
    n_jobs=-1,
)
grid.fit(X_tr, y_tr)

print(f"\nBest params: {grid.best_params_}")
print(f"Best CV BalAcc: {grid.best_score_:.4f}")

# Evaluate on held-out test split
best_rf = grid.best_estimator_
y_pred = best_rf.predict(X_te)
y_proba = best_rf.predict_proba(X_te)[:, 1]

tune_balacc = balanced_accuracy_score(y_te, y_pred)
tune_auc    = roc_auc_score(y_te, y_proba)
tune_oob    = 1 - best_rf.oob_score_

print(f"\nTest set (tuning split):")
print(f"  BalAcc:  {tune_balacc:.4f}")
print(f"  AUC:     {tune_auc:.4f}")
print(f"  OOB err: {tune_oob:.4f}")

# Store best params for later sections
BEST_PARAMS = grid.best_params_
del X_tr, X_te, y_tr, y_te  # free memory

# Plot GridSearch results summary
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

# Extract CV results
cv_res = pd.DataFrame(grid.cv_results_)
# For each of the 4 params, show mean score vs that param
for i, param in enumerate(["param_n_estimators", "param_max_depth",
                            "param_min_samples_leaf", "param_class_weight"]):
    ax = axes[i]
    if param == "param_max_depth":
        vals = cv_res[param].apply(lambda x: str(x) if pd.notna(x) else "None")
        unique = [str(x) if x is not None else "None" for x in param_grid["max_depth"]]
    elif param == "param_class_weight":
        vals = cv_res[param].apply(lambda x: str(x))
        unique = [str(x) for x in param_grid["class_weight"]]
    elif param == "param_n_estimators":
        vals = pd.to_numeric(cv_res[param])
        unique = param_grid["n_estimators"]
    elif param == "param_min_samples_leaf":
        vals = pd.to_numeric(cv_res[param])
        unique = param_grid["min_samples_leaf"]

    means = [cv_res.loc[cv_res[param].apply(str) == str(u), "mean_test_score"].mean()
             for u in unique]
    stds  = [cv_res.loc[cv_res[param].apply(str) == str(u), "mean_test_score"].std()
             for u in unique]
    x_pos = range(len(unique))
    ax.bar(x_pos, means, yerr=stds, capsize=5, color="#1f77b4", alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(unique, fontsize=8, rotation=20 if param == "param_class_weight" else 0)
    ax.set_ylabel("Mean CV BalAcc")
    name = param.replace("param_", "")
    ax.set_title(name)
    ax.axhline(grid.best_score_, color="#d62728", ls="--", alpha=0.5, linewidth=0.8)

plt.suptitle(f"GridSearchCV — {DRUG} / {SPECIES}", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "grid_search.pdf", bbox_inches="tight")
plt.show()

---
## 4. CROSS-SITE EVALUATION — TRAIN ON A, TEST ON B/C/D

In [ ]:
# =============================================================================
# 4.1  Train on DRIAMS-A with best params, test on B, C, D
# =============================================================================

X_A = site_data["A"]["X"]
y_A = site_data["A"]["y"]

rf_cross = RandomForestClassifier(**BEST_PARAMS, oob_score=True, random_state=SEED_TUNE, n_jobs=-1)
rf_cross.fit(X_A, y_A)

print(f"Trained on DRIAMS-A: n={len(y_A)}  OOB error={1 - rf_cross.oob_score_:.4f}")
print()

# Evaluate on each site (including A as baseline)
cross_results = {}
for site in "ABCD":
    X_site = site_data[site]["X"]
    y_site = site_data[site]["y"]

    preds = rf_cross.predict(X_site)
    proba = rf_cross.predict_proba(X_site)[:, 1]

    balacc = balanced_accuracy_score(y_site, preds)
    auc    = roc_auc_score(y_site, proba)

    cross_results[site] = {"BalAcc": balacc, "AUC": auc, "n": len(y_site)}

    label = " (train)" if site == "A" else ""
    print(f"  DRIAMS-{site}{label}: n={len(y_site):4d}  BalAcc={balacc:.4f}  AUC={auc:.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4.5))
sites = list(cross_results.keys())
x = np.arange(len(sites))
w = 0.3
balaccs = [cross_results[s]["BalAcc"] for s in sites]
aucs    = [cross_results[s]["AUC"]    for s in sites]
ax.bar(x - w/2, balaccs, w, label="BalAcc", color="#1f77b4")
ax.bar(x + w/2, aucs,    w, label="AUC",    color="#ff7f0e")
ax.set_xticks(x)
ax.set_xticklabels([f"DRIAMS-{s}\nn={cross_results[s]['n']}" for s in sites])
ax.set_ylabel("Score")
ax.set_title(f"Cross-Site Generalization — {DRUG} / {SPECIES}\nTrained on DRIAMS-A")
ax.legend(loc="lower right")
ax.set_ylim(0, 1)
ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.tight_layout()
plt.savefig(OUT_DIR / "cross_site_eval.pdf", bbox_inches="tight")
plt.show()

# Confusion matrices for B, C, D
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, site in zip(axes, "BCD"):
    ConfusionMatrixDisplay.from_predictions(
        site_data[site]["y"],
        rf_cross.predict(site_data[site]["X"]),
        display_labels=["Susceptible", "Resistant"],
        cmap="Blues", ax=ax, colorbar=False,
    )
    ax.set_title(f"DRIAMS-{site} (n={site_data[site]['n']})", fontsize=12)
fig.suptitle(f"Cross-Site Confusion Matrices — {DRUG} / {SPECIES}", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "cross_site_confusion.pdf", bbox_inches="tight")
plt.show()

---
## 5. MULTI-SEED POOLED EVALUATION — 5× 75/25 Splits

In [ ]:
# =============================================================================
# 5.1  Pooled A+B+C+D, 5 random stratified splits with different seeds
# =============================================================================

SEEDS = [42, 123, 456, 789, 1011]
multi_results = []

print("Multi-seed evaluation on pooled A+B+C+D:")
print(f"  Best params: {BEST_PARAMS}")
print(f"  Seeds: {SEEDS}")
print()

for seed in SEEDS:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_pooled, y_pooled, test_size=0.25,
        stratify=y_pooled, random_state=seed,
    )

    rf = RandomForestClassifier(**BEST_PARAMS, oob_score=True, random_state=seed, n_jobs=-1)
    rf.fit(X_tr, y_tr)

    preds = rf.predict(X_te)
    proba = rf.predict_proba(X_te)[:, 1]

    balacc = balanced_accuracy_score(y_te, preds)
    auc    = roc_auc_score(y_te, proba)
    oob    = 1 - rf.oob_score_

    multi_results.append({
        "seed": seed, "balacc": balacc, "auc": auc, "oob": oob,
        "n_train": len(y_tr), "n_test": len(y_te),
    })

    print(f"  seed={seed:4d}  BalAcc={balacc:.4f}  AUC={auc:.4f}  OOB={oob:.4f}")

# Summary
balaccs = [r["balacc"] for r in multi_results]
aucs    = [r["auc"]    for r in multi_results]

print(f"\nPooled {DRUG} / {SPECIES} — Mean ± Std (5 seeds):")
print(f"  BalAcc:  {np.mean(balaccs):.4f} ± {np.std(balaccs):.4f}")
print(f"  AUC:     {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Bar chart with error bars
ax1.bar([0], [np.mean(balaccs)], yerr=[np.std(balaccs)],
        color="#1f77b4", capsize=5, label="BalAcc", width=0.3)
ax1.bar([1], [np.mean(aucs)],    yerr=[np.std(aucs)],
        color="#ff7f0e", capsize=5, label="AUC",    width=0.3)
ax1.set_xticks([0, 1])
ax1.set_xticklabels(["BalAcc", "AUC"])
ax1.set_ylabel("Score")
ax1.set_title(f"Pooled Performance (5 seeds) — {DRUG} / {SPECIES}")
ax1.set_ylim(0, 1)
ax1.axhline(0.5, color="gray", ls="--", alpha=0.4)

# Per-seed line plot
seeds = [r["seed"] for r in multi_results]
ax2.plot(seeds, [r["balacc"] for r in multi_results], "o-", color="#1f77b4", label="BalAcc")
ax2.plot(seeds, [r["auc"]    for r in multi_results], "s-", color="#ff7f0e", label="AUC")
ax2.set_xlabel("Random Seed")
ax2.set_ylabel("Score")
ax2.set_title("Per-Seed Consistency")
ax2.legend(loc="lower right")
ax2.set_ylim(0, 1)
ax2.axhline(0.5, color="gray", ls="--", alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR / "multi_seed_eval.pdf", bbox_inches="tight")
plt.show()

---
## 6. FEATURE IMPORTANCE — WHICH M/Z BINS DRIVE RESISTANCE PREDICTION?

In [ ]:
# =============================================================================
# 6.1  Train final RF on ALL pooled data + extract importance
# =============================================================================

rf_final = RandomForestClassifier(**BEST_PARAMS, oob_score=True, random_state=SEED_TUNE, n_jobs=-1)
rf_final.fit(X_pooled, y_pooled)

importance = rf_final.feature_importances_
oob_err_final = 1 - rf_final.oob_score_

print(f"Final RF on all pooled data ({len(y_pooled)} samples):")
print(f"  OOB error: {oob_err_final:.4f}")

# Top bins
top_n = 30
top_idx = np.argsort(importance)[-top_n:][::-1]  # descending

print(f"\nTop {top_n} most important bins:")
for rank, idx in enumerate(top_idx[:top_n], 1):
    mz = mz_axis[idx]
    imp = importance[idx]
    print(f"  {rank:2d}.  bin_{idx:4d}  m/z={mz:6.0f} Da  importance={imp:.6f}")

# Plot: full importance spectrum + top bins highlighted
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True,
                                 gridspec_kw={"height_ratios": [2, 1]})

# Full spectrum
ax1.plot(mz_axis, importance, color="#1f77b4", linewidth=0.4, alpha=0.7)
ax1.set_ylabel("Feature Importance")
ax1.set_title(f"RF Feature Importance — {DRUG} / {SPECIES} ({len(y_pooled)} samples)")

# Highlight top-30
for idx in top_idx:
    ax1.axvline(mz_axis[idx], color="#d62728", alpha=0.15, linewidth=0.5)

# Annotate top-10
for rank, idx in enumerate(top_idx[:10], 1):
    ax1.annotate(f"{mz_axis[idx]:.0f}",
                (mz_axis[idx], importance[idx]),
                fontsize=6, color="#d62728", rotation=90,
                va="bottom", ha="center")

# Zoom: top-30 as bar chart
ax2.bar(range(top_n), importance[top_idx], color="#1f77b4", alpha=0.85)
ax2.set_xticks(range(top_n))
ax2.set_xticklabels([f"{mz_axis[idx]:.0f}" for idx in top_idx],
                     rotation=90, fontsize=7)
ax2.set_xlabel("m/z (Da)")
ax2.set_ylabel("Importance")
ax2.set_title(f"Top {top_n} Most Important Bins (m/z)")

plt.tight_layout()
plt.savefig(OUT_DIR / "feature_importance.pdf", bbox_inches="tight")
plt.show()

---
## 7. REPORT

In [ ]:
report_path = Path("report.md")

lines = [
    f"# Random Forest Analysis — {DRUG} Resistance in {SPECIES}",
    "",
    f"**Drug:** {DRUG}",
    f"**Pathogen:** {SPECIES}",
    f"**Data:** DRIAMS A, B, C, D — binned MALDI-TOF spectra (6000 bins, 3 Da)",
    f"**Model:** sklearn `RandomForestClassifier`",
    "",
    "---",
    "",
    "## Data Summary",
    "",
    "| Site | Samples | Resistant | Susceptible | %R |",
    "|---|---:|---:|---:|---:|",
]

for site in "ABCD":
    d = site_data[site]
    r = (d["y"] == 1).sum()
    s = (d["y"] == 0).sum()
    lines.append(f"| DRIAMS-{site} | {d['n']} | {r} | {s} | {r/d['n']*100:.1f}% |")

lines.append(f"| **Total** | {total_n} | {total_r} | {total_n-total_r} | {total_r/total_n*100:.1f}% |")

lines += [
    "",
    "---",
    "",
    "## Best Hyperparameters (GridSearchCV)",
    "",
    f"| Parameter | Value |",
    f"|---|---|",
]
for k, v in BEST_PARAMS.items():
    lines.append(f"| {k} | `{v}` |")

lines += [
    f"",
    f"**CV Balanced Accuracy (3-fold, pooled):** {grid.best_score_:.4f}",
    "",
    "---",
    "",
    "## Cross-Site Generalization (Train A, Test B/C/D)",
    "",
    "| Site | n | BalAcc | AUC |",
    "|---|---:|---:|---:|",
]
for site in "ABCD":
    label = " (train)" if site == "A" else ""
    lines.append(f"| DRIAMS-{site}{label} | {cross_results[site]['n']} | {cross_results[site]['BalAcc']:.4f} | {cross_results[site]['AUC']:.4f} |")

lines += [
    "",
    "---",
    "",
    "## Pooled Multi-Seed Evaluation (5× 75/25 splits)",
    "",
    f"| Metric | Mean | Std |",
    f"|---|---:|---:|",
    f"| BalAcc | {np.mean(balaccs):.4f} | {np.std(balaccs):.4f} |",
    f"| AUC    | {np.mean(aucs):.4f} | {np.std(aucs):.4f} |",
    "",
    "---",
    "",
    "## Top-10 Most Important m/z Bins",
    "",
    "| Rank | Bin | m/z (Da) | Importance |",
    "|---|---:|---:|---:|",
]
for rank, idx in enumerate(top_idx[:10], 1):
    lines.append(f"| {rank} | `bin_{idx}` | {mz_axis[idx]:.0f} | {importance[idx]:.6f} |")

lines += [
    "",
    "---",
    "",
    "## Generated Figures",
    "",
    f"- `{OUT_DIR}/grid_search.pdf` — GridSearchCV per-parameter summaries",
    f"- `{OUT_DIR}/cross_site_eval.pdf` — Cross-site BalAcc/AUC bar chart",
    f"- `{OUT_DIR}/cross_site_confusion.pdf` — Confusion matrices per site",
    f"- `{OUT_DIR}/multi_seed_eval.pdf` — Multi-seed pooled evaluation",
    f"- `{OUT_DIR}/feature_importance.pdf` — Per-bin importance on m/z axis",
    "",
    f"*Report generated by `05-RandomForest-Ceftazidime-Ecoli.ipynb`*",
]

with open(report_path, "w") as f:
    f.write("\n".join(lines) + "\n")
print(f"Report saved to: {report_path.resolve()}")
print(report_path.read_text())